# Regularization: Ridge, Lasso and ElasticNet
Companion to the regularization notes.

## 1. Soft-thresholding by hand

In [ ]:
import numpy as np
def soft_threshold(w, alpha):
    return np.sign(w) * np.maximum(np.abs(w) - alpha, 0)
print(soft_threshold(np.array([3.2, -0.6]), 1.0))

## 2. Ridge in closed form, and Ridge on a singular matrix

In [ ]:
XtX = np.array([[3., 6.], [6., 14.]]); Xty = np.array([10., 23.])
for alpha in [0, 1, 10]:
    print(alpha, np.linalg.solve(XtX + alpha * np.eye(2), Xty).round(4))
singular = np.array([[1., 2.], [2., 4.]])
print("det:", np.linalg.det(singular), " with alpha=0.1 det:", np.linalg.det(singular + 0.1 * np.eye(2)).round(4))

## 3. The loan data with 40 noise columns

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ld = pd.read_csv("../../data/loan_data_train.csv").drop(columns=["ID", "Amount.Funded.By.Investors", "State"])
for col in ["Interest.Rate", "Debt.To.Income.Ratio"]:
    ld[col] = pd.to_numeric(ld[col].str.replace("%", ""), errors="coerce")
for col in ["Amount.Requested", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Employment.Length"] = pd.to_numeric(ld["Employment.Length"].str.replace("10+ years", "10", regex=False)
                                        .str.replace("< 1 year", "0", regex=False).str.extract(r"(\d+)")[0], errors="coerce")
ld = ld.drop(columns="FICO.Range").dropna()
for col in ["Home.Ownership", "Loan.Purpose"]:
    counts = ld[col].value_counts()
    ld[col] = ld[col].replace(dict.fromkeys(counts[counts < 30].index, "other"))
ld = pd.get_dummies(ld, columns=["Home.Ownership", "Loan.Purpose"], drop_first=True, dtype=float)

# 40 columns of pure noise: no relationship with the interest rate at all
rng = np.random.default_rng(0)
noise = pd.DataFrame(rng.normal(size=(len(ld), 40)), columns=[f"noise_{i}" for i in range(40)], index=ld.index)
X = pd.concat([ld.drop(columns="Interest.Rate"), noise], axis=1)
y = ld["Interest.Rate"]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2)
print(x_train.shape)

## 4. OLS vs Ridge vs Lasso with all rows and with 100 rows

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error

def make_models():
    return {"OLS": make_pipeline(StandardScaler(), LinearRegression()),
            "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 100), cv=5)),
            "Lasso": make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0))}

for n in [len(x_train), 100]:                      # all 1475 training rows, then only 100
    print(f"--- {n} training rows ---")
    models = make_models()
    for name, m in models.items():
        m.fit(x_train.iloc[:n], y_train.iloc[:n])
        print(f"{name:6s} train MAE {mean_absolute_error(y_train.iloc[:n], m.predict(x_train.iloc[:n])):.3f}"
              f"   test MAE {mean_absolute_error(y_test, m.predict(x_test)):.3f}")

## 5. What happened to the noise columns (100-row models)

In [ ]:
is_noise = x_train.columns.str.startswith("noise_")
ols_w = models["OLS"][-1].coef_
lasso = models["Lasso"][-1]
ridge = models["Ridge"][-1]
print("chosen alphas: Ridge", round(ridge.alpha_, 3), " Lasso", round(lasso.alpha_, 4))
print("mean |weight| on noise columns: OLS", np.abs(ols_w[is_noise]).mean().round(4),
      " Ridge", np.abs(ridge.coef_[is_noise]).mean().round(4), " Lasso", np.abs(lasso.coef_[is_noise]).mean().round(4))
print("noise columns set to exactly 0 by Lasso:", int((lasso.coef_[is_noise] == 0).sum()), "of 40")
print("real columns set to 0 by Lasso:", [c for c, w in zip(x_train.columns[~is_noise], lasso.coef_[~is_noise]) if w == 0])

## 6. The Lasso path

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso

Xs = StandardScaler().fit_transform(x_train)
alphas = np.logspace(-3, 0.5, 60)
path = np.array([Lasso(alpha=a, max_iter=20000).fit(Xs, y_train).coef_ for a in alphas])
for j, name in enumerate(x_train.columns):
    plt.plot(alphas, path[:, j], color="lightgray" if name.startswith("noise") else None,
             lw=0.8 if name.startswith("noise") else 1.5, label=None if name.startswith("noise") else name)
plt.xscale("log"); plt.xlabel("alpha (penalty strength)"); plt.ylabel("weight (standardised features)")
plt.title("Lasso path: noise columns (grey) are the first to hit zero")

## Try it yourself
1. Soft-threshold `[3.2, -0.6, 0.45, -1.1]` with alpha 0.5.
2. Rerun section 4 with 200 and 400 rows.
3. `ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9])` on the 100-row data: which mix wins, how many features survive?